In [ ]:
import os
import pandas as pd
file_path = r'C:\project\political_ner\Research notes(1-1608) (3).xlsx'
research_notes_df = pd.read_excel(file_path)

In [ ]:
# research_notes_df = research_notes_df.rename(columns={
#     'What are the most prominent people or parties you have seen on TikTok today?': 'accountseen_tiktok',
#     'Have you seen any political actors belonging to some political groups in the European Parliament on TikTok today?\n': 'visibility_tiktok',
#     'What are the most prominent people or parties you have seen on Instagram today?  ': 'accountseen_instagram',
#     'Have you seen any politicians from some political groups in the European Parliament on Instagram today?\n': 'visibility_instagram',
# })

research_notes_df = research_notes_df.rename(columns={
    'Please list the names of official or affiliated accounts where you have seen these people or parties:\n': 'accountnames_tiktok',
    'Please list the names of official or affiliated accounts where you have seen these people or parties:': 'accountnames_instagram',
})


research_notes_df.columns


In [ ]:
research_notes_df['accountnames_tiktok']

In [ ]:
import pandas as pd
import pycountry
import re

file_path = r'C:\project\political_ner\Research notes(1-1608) (3).xlsx'
research_notes_df = pd.read_excel(file_path)


research_notes_df = research_notes_df.rename(columns={
    'Please list the names of official or affiliated accounts where you have seen these people or parties:\n': 'accountnames_tiktok',
    'Please list the names of official or affiliated accounts where you have seen these people or parties:': 'accountnames_instagram',
})


replacement_dict = {
    ' ': '',
    'DE§': 'DE3',
    'BG4': 'BG3',
    'F13': 'FI3',
    'HRO': 'HR2',
    'HRS': 'HR2',
    'HU0': 'HU3',
}
col_name = 'Your identifier (e.g. PT1, FR3, BG2)\n'

research_notes_df[col_name] = research_notes_df[col_name].replace(replacement_dict)

research_notes_df['country code'] = research_notes_df[col_name].str[:2]

def get_country_name(code):
    try:
        country = pycountry.countries.get(alpha_2=code.upper())
        if country:
            return country.name
    except:
        pass
    return 'Unknown'

research_notes_df['country'] = research_notes_df['country code'].apply(get_country_name)

research_notes_df['id_variable'] = research_notes_df['ID'].astype(str) + '_' + research_notes_df['country']

def sanitize_filename(name):
    return re.sub(r'[\\/*?:"<>|]', "", name)

column_mapping = {}
for col in research_notes_df.columns:
    simplified_col = col.strip().replace('\n', '').replace('\xa0', '').lower()
    column_mapping[simplified_col] = col

simplified_columns_to_merge = [
    'accountnames_tiktok',
    'accountnames_instagram'
]

columns_to_merge = [column_mapping.get(col, None) for col in simplified_columns_to_merge]

missing_columns = [simplified_columns_to_merge[i] for i, col in enumerate(columns_to_merge) if col is None]
if missing_columns:
    print("Could not find the following columns in the DataFrame:")
    print(missing_columns)
else:
    print("All columns matched successfully.")

if not missing_columns:
    countries = research_notes_df['country'].unique()

    for country in countries:
        df_country = research_notes_df[research_notes_df['country'] == country]
        safe_country_name = sanitize_filename(country)
        var_name = 'df_' + safe_country_name.replace(' ', '_')

        melted_df = pd.melt(
            df_country,
            id_vars=['id_variable'],
            value_vars=columns_to_merge,
            var_name='variable',
            value_name='merged'
        )

        melted_df['id_variable_column'] = melted_df['id_variable'] + '_' + melted_df['variable']

        result_df = melted_df[['id_variable_column', 'merged']]

        result_df = result_df.dropna(subset=['merged'])


        result_df['merged'] = result_df['merged'].str.replace(r'[;,]\s*', ' , ', regex=True)

        merged_var_name = var_name + '_merged'
        globals()[merged_var_name] = result_df

        result_df.to_excel(f"C:\\project\\political_ner\\Insta_TikTok_AccountNames\\{safe_country_name}_accountnames.xlsx", index=False)




        print(f"Processed data for {safe_country_name}")

else:
    print("Please adjust the 'simplified_columns_to_merge' list to match your DataFrame columns.")


In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline

model_name = "xlm-roberta-large-finetuned-conll03-english"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForTokenClassification.from_pretrained(model_name)

nlp = pipeline("ner", model=model, tokenizer=tokenizer, aggregation_strategy="simple")

In [ ]:
countries = [
    "Portugal", "Unknown", "Finland", "Hungary", "Spain", "Poland", 
    "Sweden", "Germany", "Croatia", "Bulgaria", "France"
]

input_dir = r'C:\project\political_ner\Insta_TikTok_AccountNames'
output_dir = r'C:\project\political_ner\Insta_TikTok_AccountNames\NER_Identify'

for country in countries:
    try:
        input_file = f"{input_dir}/{country}_accountnames.xlsx"
        output_file = f"{output_dir}/{country}_NER.xlsx"

        # Load the dataset for the country
        df_country = pd.read_excel(input_file)

        results = []
        for index, row in df_country.iterrows():
            id_variable_column = row.get('id_variable_column', None)
            original_text = row.get('merged', None)

            if pd.notnull(original_text) and str(original_text).strip():
                entities = nlp(str(original_text))

                for entity in entities:
                    entity_type = entity.get('entity_group', 'N/A')
                    if entity_type in ['ORG', 'PER']:
                        entity_text = entity['word']
                        # Append the result to the list
                        results.append({
                            'id_variable_column': id_variable_column,
                            'original': original_text,
                            'NER': entity_text
                        })
            else:
                pass  

        df_country_NER = pd.DataFrame(results)

        df_country_NER.to_excel(output_file, index=False)

        print(f"Processed and saved NER results for {country}")

    except Exception as e:
        print(f"Error processing {country}: {e}")


In [ ]:
import os
import pandas as pd

# List of countries
countries = [
    "Portugal", "Unknown", "Finland", "Hungary", "Spain", "Poland", 
    "Sweden", "Germany", "Croatia", "Bulgaria", "France"
]

# Input and output directories/paths
input_dir = r'C:\project\political_ner\NER_Results'
output_file = r'C:\project\political_ner\acc_names.xlsx'

df_list = []

for country in countries:
    # Construct the file path for each country's Excel file
    file_path = os.path.join(input_dir, f"{country}_NER.xlsx")
    
    # Check if the file exists before attempting to read
    if not os.path.isfile(file_path):
        print(f"File not found for {country}: {file_path}")
        continue
    
    # Read the file into a DataFrame
    df = pd.read_excel(file_path)
    
    # Add a new column with the country name
    df['Country'] = country
    
    # Append this DataFrame to our list
    df_list.append(df)

# Combine all DataFrames into one
if df_list:
    combined_df = pd.concat(df_list, ignore_index=True)
    
    # Save the combined DataFrame to the specified output path
    combined_df.to_excel(output_file, index=False)
    print(f"Combined DataFrame saved to {output_file}")
else:
    print("No files were combined. Please check that all files exist and try again.")


In [ ]:
# import pandas as pd
# import requests

# input_file = r'C:\project\political_ner\acc_names.xlsx'
# df = pd.read_excel(input_file)

# api_key = os.environ["OPENAI_API_KEY"] 

# url = 'https://api.openai.com/v1/chat/completions'
# headers = {
#     'Content-Type': 'application/json',
#     'Authorization': f'Bearer {api_key}'
# }

# country_groups = df.groupby('Country')['NER'].apply(lambda x: x.dropna().unique())
# all_unique_ner = set()
# for country, ners in country_groups.items():
#     all_unique_ner.update(ners)

# def classify_ner(ner_value):
#     messages = [
#         {"role": "system", "content": "You are a classifier that decides if a given text is likely a social media handle or a person's name."},
#         {"role": "user", "content": f"The text is: '{ner_value}'\n\nIf this text is likely a TikTok or Instagram handle, respond with '1'. If it is more like a normal person's name, respond with '0'. Don't provide any explanations, just '0' or '1'."}
#     ]

#     data = {
#         "model": "gpt-4o-mini",
#         "messages": messages,
#         "max_tokens": 1,
#         "temperature": 0
#     }

#     response = requests.post(url, headers=headers, json=data)
#     if response.status_code == 200:
#         result = response.json()
#         choice = result['choices'][0]['message']['content'].strip()
#         if choice not in ['0', '1']:
#             # fallback to '0' if unexpected response
#             choice = '0'
#         return int(choice)
#     else:
#         # In case of error, default to 0
#         return 0

# # Classify all unique NER values
# classification_dict = {}
# for ner_val in all_unique_ner:
#     classification_dict[ner_val] = classify_ner(ner_val)

# # We'll now write results to a text file
# output_file = r'C:\project\political_ner\acc_names_account_list.txt'

# with open(output_file, 'w', encoding='utf-8') as f:
#     for country, ners in country_groups.items():
#         # Filter ners that are classified as '1'
#         country_accounts = [ner for ner in ners if classification_dict.get(ner, 0) == 1]

#         if len(country_accounts) > 0:
#             f.write("#########################################################\n")
#             f.write(f"{country}\n")
#             for acc in country_accounts:
#                 f.write(f"{acc}\n")
#             f.write("#########################################################\n\n")

# print(f"Finished writing classified account names to {output_file}")


In [ ]:
import pandas as pd
import requests
import os

# Load the DataFrame
input_file = r'C:\project\political_ner\acc_names.xlsx'
df = pd.read_excel(input_file)

# Your OpenAI API key
api_key = os.environ["OPENAI_API_KEY"] 
url = 'https://api.openai.com/v1/chat/completions'
headers = {
    'Content-Type': 'application/json',
    'Authorization': f'Bearer {api_key}'
}

# Extract tokens by country
country_tokens = {}  # dict: country -> set of tokens
for idx, row in df.iterrows():
    country = row['Country']
    original_text = row.get('original', '')
    if pd.isna(original_text):
        continue
    
    # Split the text by whitespace to get tokens
    tokens = original_text.split()
    
    if country not in country_tokens:
        country_tokens[country] = set()
    
    for tok in tokens:
        # Add the token to that country's set
        country_tokens[country].add(tok)

# Combine all tokens from all countries for classification
all_tokens = set()
for c, toks in country_tokens.items():
    all_tokens.update(toks)

# Classification function
def classify_token(token):
    messages = [
        {"role": "system", "content": "You are a classifier that decides if a given text is likely a social media handle or a person's name."},
        {"role": "user", "content": f"The text is: '{token}'\n\nIf this text is likely a TikTok or Instagram handle, respond with '1'. If it is more like a normal person's name or not a handle, respond with '0'. Don't provide any explanations, just '0' or '1'."}
    ]

    data = {
        "model": "gpt-4o-mini",
        "messages": messages,
        "max_tokens": 1,
        "temperature": 0
    }

    response = requests.post(url, headers=headers, json=data)
    if response.status_code == 200:
        result = response.json()
        choice = result['choices'][0]['message']['content'].strip()
        if choice not in ['0', '1']:
            choice = '0'
        return int(choice)
    else:
        # In case of error, default to 0
        return 0

classification_dict = {}
total_tokens = len(all_tokens)
print(f"Total unique tokens to classify: {total_tokens}")

for i, token in enumerate(all_tokens, start=1):
    classification_dict[token] = classify_token(token)
    if i % 100 == 0:
        print(f"Progress: Classified {i}/{total_tokens} tokens...")

print("Classification completed for all tokens. Now writing results to file...")

output_file = r'C:\project\political_ner\acc_names_tiktok.txt'

with open(output_file, 'w', encoding='utf-8') as f:
    for country, toks in country_tokens.items():
        # Get all tokens for this country that are classified as accounts
        country_accounts = [t for t in toks if classification_dict.get(t, 0) == 1]

        if country_accounts:
            f.write("#########################################################\n")
            f.write(f"{country}\n")
            for acc in country_accounts:
                f.write(f"{acc}\n")
            f.write("#########################################################\n\n")

print(f"Finished writing classified account names to {output_file}")
